### **Transfer Learning Framework** 
(Foundational Model)

In [ ]:
TARGET_CONDITIONS = [
    # ───────────────────────────────────────
    # PHASE 0: FOUNDATION (Preceded the log file)
    # ───────────────────────────────────────
    'Autism Spectrum Disorder',                     # n=585 (Foundation)
    'NervousSystem_Dementia_Developmental',         # n=122
    'Psychopathology_Dementia',                     # n=98
    'Psychopathology_Organic_Mental_Disorder',      # n=180
    
    # ───────────────────────────────────────
    # PHASE 1: THE LOG FILE STARTS HERE
    # (Loading weights from Organic_Mental_Disorder)
    # ───────────────────────────────────────
    'NervousSystem_Cerebrovascular',                # n=592 (✅ Valid)
    'NervousSystem_Inflammatory_Infectious',        # n=92  (✅ Valid)
    'ICD_F31_Bipolar',                              # n=110 (✅ Valid)
    'Psychopathology_Schizophrenia_Spectrum',       # n=66  (✅ Valid)
    
    # ───────────────────────────────────────
    # PHASE 2: THE "GRAVEYARD" (Failed in log, weights discarded)
    # ───────────────────────────────────────
    'ICD_F32_Depressive_Episode',                   # n=4,338 (❌ Failed)
    'NervousSystem_Sleep_Disorders',                # n=1,104 (❌ Failed)
    'Psychopathology_Substance_Use',                # n=2,276 (❌ Failed)
    
    # ───────────────────────────────────────
    # PHASE 3: THE RECOVERY (Skipped failures, loaded from Schizophrenia)
    # ───────────────────────────────────────
    'NervousSystem_Epilepsy_Status_Epilepticus',    # n=342 (✅ Valid)
    'NervousSystem_Parkinsons_Other_Movement',      # n=416 (✅ Valid)
    'NervousSystem_Multiple_Sclerosis_Other_Demyelinating', # n=164 (✅ Valid)
]

In [ ]:
# Import necessary libraries
import os
from pathlib import Path
from typing import List, Optional, Dict, Any
from bbtransformer import run_analysis


# Transfer learning pipeline
class BBTransformerAnalyzer:
    def __init__(
        self,
        base_dir: str,
        weights_dir: str = "weights",
        results_dir: str = "results",
        initial_weights: Optional[str] = None,
        min_composite: float = 0.60,
        max_trials_per_disorder: int = 50
    ):
        self.base_dir = Path(base_dir)
        self.weights_dir = Path(weights_dir)
        self.results_dir = Path(results_dir)
        self.weights_dir.mkdir(exist_ok=True)
        self.results_dir.mkdir(exist_ok=True)
        
        self.current_weights = initial_weights
        self.min_composite = min_composite
        self.max_trials = max_trials_per_disorder
        self.valid_models = []  # Track successful disorders

    def get_paths(self, disorder: str):
        return (
            self.base_dir / f"fmri_{disorder}.npz",
            self.base_dir / f"pheno_{disorder}.csv"
        )

    def is_valid(self, metrics: Dict[str, float]) -> bool:
        """Check if all core metrics meet clinical threshold."""
        return all(
            metrics.get(metric, 0) >= self.min_composite
            for metric in ['f1', 'roc_auc', 'accuracy', 'precision', 'recall']
        )

    def run_ordered_pipeline(self, disorders: List[str]) -> Dict[str, Any]:
        """
        Run transfer learning in strict biological order.
        Propagate weights from last VALID model, even if intermediate disorders fail.
        """
        results_summary = {}

        for i, disorder in enumerate(disorders, 1):
            print(f"\n{'='*70}")
            print(f"PHASE {i}/{len(disorders)}: {disorder}")
            print(f"{'='*70}")

            data_path, pheno_path = self.get_paths(disorder)
            
            # Use pretrained weights if we have ANY valid model so far
            use_pretrained = self.current_weights is not None
            
            # Try up to max_trials to find a valid model
            best_result = None
            for trial in range(self.max_trials):
                print(f"  Trial {trial+1}/{self.max_trials}...")

                BEST_PARAMS = {
                    'embed_dim': 512,
                    'num_heads': 16,
                    'num_layers': 7,
                    'n_kv_heads': 4,              
                    'dropout_input': 0.18,      
                    'dropout_patch': 0.16,      
                    'dropout_attn': 0.15,       
                    'dropout_ffn': 0.25,        
                    'dropout_classifier': 0.07,  
                    'dropout_temporal': 0.16,   
                    'embed_dim_age': 32,
                    'embed_dim_ext': 16,
                    'patch_size': 3,
                    'patch_embed_ratio': 0.75,
                    'temp_attn_hidden': 512,
                    'return_attn_weights': False,
                    'stochastic_depth_rate': 0.07
                }

                TRAIN_PARAMS = {
                    'epochs': 5000,
                    'lr': 2.3157e-05,            
                    'weight_decay': 1.14e-06,     
                    'patience': 90
                }                
                
                try:
                    result = run_analysis(
                        model_config=BEST_PARAMS,
                        training_config=TRAIN_PARAMS,
                        target_column=disorder,
                        data_path=str(data_path),
                        pheno_path=str(pheno_path),
                        use_pretrained=use_pretrained,
                        pretrained_weight_file=self.current_weights,
                        compute_importance=False,
                        random_seed=42 + trial,
                        weights_dir=str(self.weights_dir)
                    )
                    
                    if self.is_valid(result['metrics']):
                        best_result = result
                        print(f"  ✅ VALID MODEL FOUND (Composite: {result['metrics']['f1']:.4f})")
                        break
                    else:
                        print(f"  ❌ Trial {trial+1} failed validity check")
                        
                except Exception as e:
                    print(f"  ❌ Trial {trial+1} crashed: {str(e)}")
                    continue

            # Save result regardless of validity
            results_summary[disorder] = {
                'valid': best_result is not None,
                'metrics': best_result['metrics'] if best_result else None,
                'weights_used': self.current_weights,
                'weights_saved': None
            }

            # Update weights ONLY if this disorder is valid
            if best_result is not None:
                weight_file = f"weights_{disorder}.pth"
                self.current_weights = str(self.weights_dir / weight_file)
                results_summary[disorder]['weights_saved'] = self.current_weights
                self.valid_models.append(disorder)
                print(f"  🔁 Propagating weights to next disorder")
            else:
                # CRITICAL FIX: Do NOT reset weights if we have prior valid models
                if self.valid_models:
                    print(f"  ⚠️ Keeping weights from last valid model: {self.valid_models[-1]}")
                    # self.current_weights remains UNCHANGED
                else:
                    print(f"  🧼 No prior valid model—next disorder will train from scratch")
                    self.current_weights = None

        return results_summary

In [ ]:
# Run the script

analyzer = BBTransformerAnalyzer(
    base_dir='/mnt/movement/users/jaizor/xtra/data/fmri/chrt',
    weights_dir='/mnt/movement/users/jaizor/xtra/ΞΞ/Training/Test2/weights',
    initial_weights='weights_Psychopathology_Organic_Mental_Disorder.pth',  
    min_composite=0.60,
    max_trials_per_disorder=15
)

results = analyzer.run_ordered_pipeline(TARGET_CONDITIONS)

# Print success summary
print("\n✅ VALID MODELS:")
for disorder in analyzer.valid_models:
    print(f"  - {disorder}")


PHASE 1/10: NervousSystem_Cerebrovascular
  Trial 1/15...
STEP 1: Loading Data for Target = 'NervousSystem_Cerebrovascular'
  fMRI: /mnt/movement/users/jaizor/xtra/data/fmri/chrt/fmri_NervousSystem_Cerebrovascular.npz
  Phenotype: /mnt/movement/users/jaizor/xtra/data/fmri/chrt/pheno_NervousSystem_Cerebrovascular.csv
Loaded phenotype: (592, 56)
Loaded fMRI: (592, 150, 414)
  Subjects: 592
  Timepoints: 150
  Brain regions: 414
  ROI labels: 414
All required columns present

STEP 2: Preparing Data Loaders
Cohort: 592 subjects (296 cases, 296 controls, 50.0% prevalence)
Splits → Train: 414, Val: 89, Test: 89

Dataset Meta
  target: NervousSystem_Cerebrovascular
  n_total: 592
  n_positive: 296
  prevalence: 0.5
  feature_dim: 414
  n_train: 414
  n_val: 89
  n_test: 89

STEP 3: Initializing BBTransformer
Model created on cuda with 29,354,880 parameters

STEP 3.5: Loading Pretrained Weights
  From: /mnt/movement/users/jaizor/xtra/ΞΞ/Training/Test2/weights/weights_Psychopathology_Organic_M

Early stopping at epoch 248 (F1: 0.4578)
✓ Saved SAFE weights (weights_only=True compatible) to: weights/weights_NervousSystem_Cerebrovascular.pth

STEP 5: Test Set Evaluation



Performance Metrics:
  Accuracy:  0.5393
  Precision: 0.5205
  Recall:    0.8636
  F1 Score:  0.6496
  ROC-AUC:   0.5452

Confusion Matrix:
[[10 35]
 [ 6 38]]

[INFO] Skipping permutation importance (compute_importance=False).
Results saved to JSON: results/NervousSystem_Cerebrovascular_results.json

TRAINING & EVALUATION COMPLETE
Target: NervousSystem_Cerebrovascular
  ❌ Trial 1 failed validity check
  Trial 2/15...
STEP 1: Loading Data for Target = 'NervousSystem_Cerebrovascular'
  fMRI: /mnt/movement/users/jaizor/xtra/data/fmri/chrt/fmri_NervousSystem_Cerebrovascular.npz
  Phenotype: /mnt/movement/users/jaizor/xtra/data/fmri/chrt/pheno_NervousSystem_Cerebrovascular.csv
Loaded phenotype: (592, 56)
Loaded fMRI: (592, 150, 414)
  Subjects: 592
  Timepoints: 150
  Brain regions: 414
  ROI labels: 414
All required columns present

STEP 2: Preparing Data Loaders
Cohort: 592 subjects (296 cases, 296 controls, 50.0% prevalence)
Splits → Train: 414, Val: 89, Test: 89

Dataset Meta
  target:

Early stopping at epoch 186 (F1: 0.5962)
✓ Saved SAFE weights (weights_only=True compatible) to: weights/weights_NervousSystem_Cerebrovascular.pth

STEP 5: Test Set Evaluation



Performance Metrics:
  Accuracy:  0.5730
  Precision: 0.5429
  Recall:    0.8636
  F1 Score:  0.6667
  ROC-AUC:   0.5960

Confusion Matrix:
[[13 32]
 [ 6 38]]

[INFO] Skipping permutation importance (compute_importance=False).
Results saved to JSON: results/NervousSystem_Cerebrovascular_results.json

TRAINING & EVALUATION COMPLETE
Target: NervousSystem_Cerebrovascular
  ❌ Trial 2 failed validity check
  Trial 3/15...
STEP 1: Loading Data for Target = 'NervousSystem_Cerebrovascular'
  fMRI: /mnt/movement/users/jaizor/xtra/data/fmri/chrt/fmri_NervousSystem_Cerebrovascular.npz
  Phenotype: /mnt/movement/users/jaizor/xtra/data/fmri/chrt/pheno_NervousSystem_Cerebrovascular.csv
Loaded phenotype: (592, 56)
Loaded fMRI: (592, 150, 414)
  Subjects: 592
  Timepoints: 150
  Brain regions: 414
  ROI labels: 414
All required columns present

STEP 2: Preparing Data Loaders
Cohort: 592 subjects (296 cases, 296 controls, 50.0% prevalence)
Splits → Train: 414, Val: 89, Test: 89

Dataset Meta
  target:

...


Performance Metrics:
  Accuracy:  0.6538
  Precision: 0.6333
  Recall:    0.7308
  F1 Score:  0.6786
  ROC-AUC:   0.6672

Confusion Matrix:
[[15 11]
 [ 7 19]]

[INFO] Skipping permutation importance (compute_importance=False).
Results saved to JSON: results/NervousSystem_Epilepsy_Status_Epilepticus_results.json

TRAINING & EVALUATION COMPLETE
Target: NervousSystem_Epilepsy_Status_Epilepticus
  ✅ VALID MODEL FOUND (Composite: 0.6786)
  🔁 Propagating weights to next disorder

PHASE 9/10: NervousSystem_Parkinsons_Other_Movement
  Trial 1/15...
STEP 1: Loading Data for Target = 'NervousSystem_Parkinsons_Other_Movement'
  fMRI: /mnt/movement/users/jaizor/xtra/data/fmri/chrt/fmri_NervousSystem_Parkinsons_Other_Movement.npz
  Phenotype: /mnt/movement/users/jaizor/xtra/data/fmri/chrt/pheno_NervousSystem_Parkinsons_Other_Movement.csv
Loaded phenotype: (416, 56)
Loaded fMRI: (416, 150, 414)
  Subjects: 416
  Timepoints: 150
  Brain regions: 414
  ROI labels: 414
All required columns present

ST

Early stopping at epoch 94 (F1: 0.7246)
✓ Saved SAFE weights (weights_only=True compatible) to: weights/weights_NervousSystem_Parkinsons_Other_Movement.pth

STEP 5: Test Set Evaluation



Performance Metrics:
  Accuracy:  0.7302
  Precision: 0.7333
  Recall:    0.7097
  F1 Score:  0.7213
  ROC-AUC:   0.7782

Confusion Matrix:
[[24  8]
 [ 9 22]]

[INFO] Skipping permutation importance (compute_importance=False).
Results saved to JSON: results/NervousSystem_Parkinsons_Other_Movement_results.json

TRAINING & EVALUATION COMPLETE
Target: NervousSystem_Parkinsons_Other_Movement
  ✅ VALID MODEL FOUND (Composite: 0.7213)
  🔁 Propagating weights to next disorder

PHASE 10/10: NervousSystem_Multiple_Sclerosis_Other_Demyelinating
  Trial 1/15...
STEP 1: Loading Data for Target = 'NervousSystem_Multiple_Sclerosis_Other_Demyelinating'
  fMRI: /mnt/movement/users/jaizor/xtra/data/fmri/chrt/fmri_NervousSystem_Multiple_Sclerosis_Other_Demyelinating.npz
  Phenotype: /mnt/movement/users/jaizor/xtra/data/fmri/chrt/pheno_NervousSystem_Multiple_Sclerosis_Other_Demyelinating.csv
Loaded phenotype: (164, 56)
Loaded fMRI: (164, 150, 414)
  Subjects: 164
  Timepoints: 150
  Brain regions: 414
 

Early stopping at epoch 91 (F1: 0.5000)
✓ Saved SAFE weights (weights_only=True compatible) to: weights/weights_NervousSystem_Multiple_Sclerosis_Other_Demyelinating.pth

STEP 5: Test Set Evaluation



Performance Metrics:
  Accuracy:  0.8400
  Precision: 0.8333
  Recall:    0.8333
  F1 Score:  0.8333
  ROC-AUC:   0.9167

Confusion Matrix:
[[11  2]
 [ 2 10]]

[INFO] Skipping permutation importance (compute_importance=False).
Results saved to JSON: results/NervousSystem_Multiple_Sclerosis_Other_Demyelinating_results.json

TRAINING & EVALUATION COMPLETE
Target: NervousSystem_Multiple_Sclerosis_Other_Demyelinating
  ✅ VALID MODEL FOUND (Composite: 0.8333)
  🔁 Propagating weights to next disorder

✅ VALID MODELS:
  - NervousSystem_Cerebrovascular
  - NervousSystem_Inflammatory_Infectious
  - ICD_F31_Bipolar
  - Psychopathology_Schizophrenia_Spectrum
  - NervousSystem_Epilepsy_Status_Epilepticus
  - NervousSystem_Parkinsons_Other_Movement
  - NervousSystem_Multiple_Sclerosis_Other_Demyelinating
